# 04 · Directions and the decomposition of δ

The four directions, all `[29, H]` with index 0 the embedding:

| name | definition | who could obtain it |
|---|---|---|
| `delta_mixed` | `unit(mean_mix − mean_base)` | a real auditor |
| `delta_clean` | `unit(mean_clean − mean_base)` | an auditor who trains one clean reference student |
| `delta_iso` | `unit(mean_mix − mean_clean)` | oracle: the trait term, isolated |
| `delta_pureA` | `unit(mean_pureA − mean_base)` | oracle ceiling |

`build_directions` already computes all four (`realistic`, `generic`,
`oracle_matched`, `pureA`); this notebook only renames them to the PLAN v2
vocabulary. No new direction code.

**C2** is the claim that `delta_mixed` is dominated by the domain term. Testing
it needs the **raw** means — `‖δ_iso‖ / ‖δ_mixed‖` is a ratio of magnitudes, and
unit-normalization is exactly what destroys it. So the decomposition table is
computed separately from the (normalized) scoring directions, and `delta_iso` is
always `mixed − clean` on raw means, never `unit(δ_mixed) − unit(δ_clean)`.

In [ ]:
# --- bootstrap: identical first cell in every pivot notebook -----------------
# /workspace is the Runpod network volume, so `runs/` (which config.py resolves
# relative to the repo root) survives a pod stop. Nothing here writes to the
# container disk except the HF cache, which is redirected for the same reason.
import os, sys, json, time, hashlib, subprocess
from pathlib import Path

ROOT = Path("/workspace/subliminal-attrib")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("SUBATTR_THIRD_PARTY", str(ROOT / "third_party"))
os.environ.setdefault("HF_HOME", "/workspace/hf_home")
os.environ.setdefault("WANDB_MODE", "disabled")

%load_ext autoreload
%autoreload 2

import torch
from subattr import config

cfg  = config.load("configs/pivot.yaml")
DATA = cfg.data_dir
RUN  = cfg.run_dir
MIX  = DATA / "mixtures"
T0   = time.time()

# WHICH code is running? Nothing else in this notebook would notice a `main`
# checkout until a missing file several cells in, and pivot and main answer
# different questions -- their results have to stay independently attributable.
# sys.path puts ROOT/src first so the working tree beats any installed copy;
# assert that rather than assume it.
BRANCH = subprocess.run(
    ["git", "-C", str(ROOT), "rev-parse", "--abbrev-ref", "HEAD"],
    capture_output=True, text=True,
).stdout.strip()
assert BRANCH == "pivot", f"expected the 'pivot' branch at {ROOT}, found {BRANCH!r}"
assert Path(config.__file__).resolve().is_relative_to(ROOT / "src"), (
    f"subattr is imported from {config.__file__}, not {ROOT / 'src'}"
)
assert config.REPO_ROOT == ROOT, f"REPO_ROOT is {config.REPO_ROOT}, not {ROOT}"

print(f"config    {cfg.name}   model_hash={cfg.hash}   data_hash={cfg.data_hash}")
print(f"branch    {BRANCH}   {config.git_sha()}")
print(f"gpu       {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")
print(f"data_dir  {DATA}")
print(f"run_dir   {RUN}")

In [ ]:
# The fraction chosen by the notebook-02 gate. Every stage after 02 reads it
# rather than hard-coding a dose, so the whole pipeline moves together if the
# gate is ever re-run.
GATE = json.loads((RUN / "gate.json").read_text())
FRACTION = GATE["fraction"]
print(f"gate fraction: {FRACTION}   (rule: {GATE['rule']})")

In [ ]:
from subattr import directions as D
from subattr import train as tr
from subattr.cache import load_tensors, save_tensors

dir_prompts = json.loads((MIX / "heldout_dirprompts.json").read_text())["prompts"]
print(f"{len(dir_prompts)} held-out direction prompts")

adapters = {
    "student_clean_matched": tr.latest_adapter(str(RUN / "students" / "clean")),
    "student_mix10": tr.latest_adapter(str(RUN / "students" / "mix10")),
    "student_mix25": tr.latest_adapter(str(RUN / "students" / "mix25")),
    "student_mix50": tr.latest_adapter(str(RUN / "students" / "mix50")),
    "student_pureA": tr.latest_adapter(str(RUN / "students" / "pureA")),
}

In [ ]:
means = D.collect_means(
    cfg.base_model, adapters, dir_prompts, protocol="svd",
    cache_path=str(RUN / "means_svd.pt"),
)
print({k: tuple(v.shape) for k, v in means.items()})
N_LAYERS = means["base"].shape[0]
print(f"{N_LAYERS} residual slots (0 = embedding, 1..{N_LAYERS - 1} = block outputs)")

In [ ]:
# Per-prompt samples for the covariance-matched null. Must reproduce the mean
# above, or the null is matched to the wrong distribution.
samples = D.collect_activation_samples(
    cfg.base_model, dir_prompts, cache_path=str(RUN / "base_samples.pt")
)
sample_mean = samples.float().mean(0)
agreement = D.cosine_per_layer(sample_mean, means["base"])
print(f"cos(mean(samples), means['base']) min over layers = {float(agreement.min()):.6f}")
assert float(agreement.min()) > 0.999, "activation samples do not reproduce the collected mean"

## 4.1 · The four directions

In [ ]:
ALIAS = {
    "realistic": "delta_mixed",
    "oracle_matched": "delta_iso",
    "generic": "delta_clean",
    "pureA": "delta_pureA",
}

def directions_for(fraction):
    subset = {
        "base": means["base"],
        "student_mixed": means[f"student_{fraction}"],
        "student_clean_matched": means["student_clean_matched"],
        "student_pureA": means["student_pureA"],
    }
    built = D.build_directions(subset, seed=cfg.seed).directions
    return {ALIAS[k]: v for k, v in built.items() if k in ALIAS}

deltas = directions_for(FRACTION)
save_tensors(deltas, RUN / "deltas.pt")
for name, v in deltas.items():
    print(f"  {name:<14s} {tuple(v.shape)}  ||.||=1 per layer (unit-normalized for scoring)")

## 4.2 · C2 — the decomposition, on raw means

In [ ]:
import pandas as pd

raw_means = {
    "base": means["base"],
    "student_mixed": means[f"student_{FRACTION}"],
    "student_clean_matched": means["student_clean_matched"],
    "student_pureA": means["student_pureA"],
}
decomp = pd.DataFrame(D.decomposition_table(raw_means))
decomp.to_csv(RUN / "decomposition.csv", index=False)
print(decomp.to_string(index=False, float_format=lambda v: f"{v:8.4f}"))

In [ ]:
LAYER_PREVIEW = 8
row = decomp[decomp.layer == LAYER_PREVIEW].iloc[0]
print(f"At layer {LAYER_PREVIEW}:")
print(f"  ||delta_mixed||          {row.norm_mixed:.4f}")
print(f"  ||delta_clean||          {row.norm_clean:.4f}")
print(f"  ||delta_iso||            {row.norm_iso:.4f}")
print(f"  ||iso|| / ||mixed||      {row.iso_over_mixed:.4f}   <- C2: small means the")
print(f"                                        domain term dominates")
print(f"  cos(delta_iso, pureA)    {row.cos_iso_pureA:.4f}   <- is the isolated term the trait?")
print(f"  cos(delta_mixed, clean)  {row.cos_mixed_clean:.4f}")
print(f"  cos(delta_mixed, pureA)  {row.cos_mixed_pureA:.4f}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].plot(decomp.layer, decomp.iso_over_mixed, marker="o", ms=3)
axes[0].set_title(r"$\|\delta_{iso}\| / \|\delta_{mixed}\|$  (C2)")
axes[0].set_xlabel("residual slot (0 = embedding)")
axes[0].axhline(0.0, color="0.7", lw=0.8)
axes[1].plot(decomp.layer, decomp.cos_iso_pureA, marker="o", ms=3, label=r"cos($\delta_{iso}$, $\delta_{pureA}$)")
axes[1].plot(decomp.layer, decomp.cos_mixed_pureA, marker="s", ms=3, label=r"cos($\delta_{mixed}$, $\delta_{pureA}$)")
axes[1].plot(decomp.layer, decomp.cos_mixed_clean, marker="^", ms=3, label=r"cos($\delta_{mixed}$, $\delta_{clean}$)")
axes[1].axhline(0.0, color="0.7", lw=0.8)
axes[1].legend(fontsize=8)
axes[1].set_xlabel("residual slot (0 = embedding)")
fig.tight_layout()
fig.savefig(RUN / "fig_decomposition.png", dpi=140)
plt.show()

## 4.3 · Dose consistency

`delta_iso` at 10%, 25% and 50% is the same contrast measured at three doses.
If those three directions do not agree with each other, "the trait direction" is
not a stable object and nothing downstream is interpretable.

In [ ]:
iso_by_fraction = {f: directions_for(f)["delta_iso"] for f in ("mix10", "mix25", "mix50")}
pairs = [("mix10", "mix25"), ("mix25", "mix50"), ("mix10", "mix50")]
print(f"{'pair':<16s} " + "  ".join(f"L{l}" for l in (4, 8, 14, 20, 27)))
for a, b in pairs:
    cos = D.cosine_per_layer(iso_by_fraction[a], iso_by_fraction[b])
    print(f"{a}-{b:<8s} " + "  ".join(f"{float(cos[l]):.3f}" for l in (4, 8, 14, 20, 27)))

print()
for f in ("mix10", "mix25", "mix50"):
    cos = D.cosine_per_layer(iso_by_fraction[f], deltas["delta_pureA"])
    print(f"cos(delta_iso[{f}], delta_pureA) at layer 8: {float(cos[8]):.3f}")

## 4.4 · Pre-registration

Layer 8 (residual index; 0 = embedding, so this is the output of block 7). Fixed
in the plan before any AUROC was computed, and written to disk so that a later
re-run cannot quietly move it. Every headline number in the report is this
layer; the 29-layer heatmap is exploratory and labelled as such.

In [ ]:
LAYER = 8
prereg_path = RUN / "preregistered_layer.json"
if prereg_path.exists():
    previous = json.loads(prereg_path.read_text())
    assert previous["layer"] == LAYER, (
        f"layer {previous['layer']} was already pre-registered on {previous['when']}; "
        f"changing it to {LAYER} after seeing results is not a pre-registration"
    )
    print(f"[pinned] layer {LAYER}, registered {previous['when']}")
else:
    prereg_path.write_text(json.dumps({
        "layer": LAYER,
        "when": time.strftime("%Y-%m-%d %H:%M:%S"),
        "git_sha": config.git_sha(),
        "why": "fixed in PLAN v2 before any AUROC was computed; mid-depth residual",
    }, indent=2))
    print(f"[registered] layer {LAYER}")

## 4.5 · The two nulls

I8: a single norm-matched Gaussian direction reached AUROC 0.82 on a dry run,
matching the best trait direction — per-example gradients are effectively
low-rank, so an isotropic draw retains real overlap with whatever separates the
sources. One draw is a sample of size one, and a Gaussian ensemble asks the
weaker of the two available questions.

* **Gaussian, n=64** — beats an arbitrary vector in ℝ³⁵⁸⁴?
* **covariance-matched, n=32** — beats an arbitrary vector *in the subspace the
  activations actually occupy*?

n=32 rather than 5 because the smallest attainable p-value is 1/(n+1), and
scoring an extra direction from the cache is one einsum.

In [ ]:
nulls = {}
nulls.update(D.random_direction_ensemble(deltas["delta_iso"], n=64, seed=cfg.seed))
nulls.update(D.covmatched_random_ensemble(samples, deltas["delta_iso"], n=32, seed=cfg.seed))
save_tensors(nulls, RUN / "nulls.pt")

gaussian = [k for k in nulls if k.startswith("random_")]
covmatched = [k for k in nulls if k.startswith("covrand_")]
print(f"{len(gaussian)} gaussian + {len(covmatched)} covariance-matched = {len(nulls)} null directions")
print(f"smallest attainable p-value: gaussian {1 / (len(gaussian) + 1):.4f}, "
      f"cov-matched {1 / (len(covmatched) + 1):.4f}")

# Both ensembles are norm-matched per layer, like the trait directions.
for name in ("random_000", "covrand_000"):
    ratio = float((nulls[name].norm(dim=-1) / deltas["delta_iso"].norm(dim=-1)).mean())
    print(f"  {name}: mean per-layer norm ratio to delta_iso = {ratio:.6f}")

In [ ]:
print(f"wall clock: {(time.time() - T0) / 60:.1f} min")

### Attended time

_Fill in before committing:_ **__ min** attended.
Copy the wall clock above and this figure into `docs/compute_log.md`.